# Tumor–T-cell Microenvironment (viva-munk port)

_Investigation `tumor-tcell-showcase` — coder reproduction notebook._

**Question.** Does the process-bigraph port of the tumor-tcell ABM — with viva-munk providing the
collision physics — reproduce the paper's mechanism, and which parts of that mechanism
hold robustly at a tractable scale (tens of cells, tens of simulated hours)?

Three replicate-seeded studies exercising the ported microenvironment: T-cell exhaustion
vs. starting PD1+ fraction, IFNg secretion (and conversion), and an in-vitro-style
killing/cytotoxicity assay — each with a mean±std population time-series and a spatial
animation.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-tumor-tcell/viva-tumor-tcell').is_dir():
    REPO = Path('/home/runner/work/viva-tumor-tcell/viva-tumor-tcell')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_tumor_tcell.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Tumor microenvironment (headline experiment) + full figure suite (`tumor-microenvironment`)

**Question.** Does the ported model reproduce tumor-tcell's headline experiment
(tumor_microenvironment_experiment, id '5') — a central tumor mass ringed by
T cells under the three CODEX conditions — and its full analysis figure suite?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` | 0 | n_tumors=60, n_tcells=12, pd1_positive_frac=0.25 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment`** — `spec_viva_tumor_tcell_composites_microenvironment_tumor_microenvironment` (a plain, editable dict)


_composite spec file for `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` not found under `viva_tumor_tcell/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tumor-microenvironment ===
STUDY = 'tumor-microenvironment'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**population_tumor**


In [ ]:
# population_tumor
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**population_tcell**


In [ ]:
# population_tcell
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**divisions**


In [ ]:
# divisions
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**deaths**


In [ ]:
# deaths
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**snapshots**


In [ ]:
# snapshots
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**spatial_animation**


In [ ]:
# spatial_animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**tumor_count_by_condition**


In [ ]:
# tumor_count_by_condition
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reproduces_headline_suite | kind=qualitative of=the 6-figure suite + cross-condition comparison |  |


## Study: T-cell exhaustion vs. starting PD1+ fraction (`tcell-exhaustion`)

**Question.** Does the exhausted (PD1+) T-cell fraction over time depend on the fraction of T cells
seeded as exhausted, and is that dependence robust across seeds?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` | 0 | n_tumors=40, n_tcells=12, pd1_positive_frac=0.75 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment`** — `spec_viva_tumor_tcell_composites_microenvironment_tumor_microenvironment` (a plain, editable dict)


_composite spec file for `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` not found under `viva_tumor_tcell/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: tcell-exhaustion ===
STUDY = 'tcell-exhaustion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**pd1p_fraction**


In [ ]:
# pd1p_fraction
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**spatial_75pct**


In [ ]:
# spatial_75pct
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| exhaustion_tracks_starting_fraction | kind=scalar of=final PD1+ T-cell fraction (25% start vs 75% start), 6 seeds |  |


## Study: IFNg secretion and PDL1n->PDL1p conversion (`phenotype-conversion`)

**Question.** Do active T cells reliably produce the IFNg field that drives tumor phenotype conversion,
and does the downstream PDL1n->PDL1p conversion itself hold at this scale?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` | 0 | n_tumors=40, n_tcells=12, pd1_positive_frac=0.25 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment`** — `spec_viva_tumor_tcell_composites_microenvironment_tumor_microenvironment` (a plain, editable dict)


_composite spec file for `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` not found under `viva_tumor_tcell/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: phenotype-conversion ===
STUDY = 'phenotype-conversion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ifng_field**


In [ ]:
# ifng_field
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**pdl1p_fraction**


In [ ]:
# pdl1p_fraction
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**spatial_conversion**


In [ ]:
# spatial_conversion
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| active_tcells_secrete_ifng | kind=scalar of=peak IFNg field (with vs without T cells), 6 seeds |  |


## Study: In-vitro-style killing / cytotoxicity assay (`killing-assay-cytotoxicity`)

**Question.** In a well-mixed assay, do active T cells produce net tumor cell death (positive
cytotoxicity) relative to a matched no-T control, robustly across seeds?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_tumor_tcell.composites.microenvironment.killing_assay` | 0 | n_tumors=30, tumor_t_ratio=1.5, pdl1_positive_frac=0.5, pd1_positive_frac=0.25 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_tumor_tcell.composites.microenvironment.killing_assay`** — `spec_viva_tumor_tcell_composites_microenvironment_killing_assay` (a plain, editable dict)


_composite spec file for `viva_tumor_tcell.composites.microenvironment.killing_assay` not found under `viva_tumor_tcell/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: killing-assay-cytotoxicity ===
STUDY = 'killing-assay-cytotoxicity'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**tumor_count_vs_control**


In [ ]:
# tumor_count_vs_control
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**population_tumor**


In [ ]:
# population_tumor
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**deaths**


In [ ]:
# deaths
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**spatial_killing**


In [ ]:
# spatial_killing
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| positive_cytotoxicity | kind=scalar of=paired cytotoxicity % across 6 seeds |  |


## Study: T-cell efficacy vs. PD1+ fraction, at larger scale (`efficacy-at-scale`)

**Question.** Does the paper's efficacy-vs-exhaustion ordering — active (25% PD1+) T cells suppress
tumor growth more than exhausted (75% PD1+) T cells — emerge when the simulation is scaled
up from the tractable 40-cell / 600-tick studies?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` | 0 | n_tumors=150, n_tcells=40, pd1_positive_frac=0.25 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment`** — `spec_viva_tumor_tcell_composites_microenvironment_tumor_microenvironment` (a plain, editable dict)


_composite spec file for `viva_tumor_tcell.composites.microenvironment.tumor_microenvironment` not found under `viva_tumor_tcell/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: efficacy-at-scale ===
STUDY = 'efficacy-at-scale'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**tumor_count_at_scale**


In [ ]:
# tumor_count_at_scale
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| efficacy_ordering_at_scale | kind=scalar of=per-seed suppression, 25% vs 75% PD1+ (150 cells, 1500 ticks, 3 seeds) |  |


## Open decisions
- Run the headline conditions at full paper scale (1200 cells, 3 days) on the mini to tighten the ~9% suppression estimate (a tens-of-hours, multi-GB job)?
- Promote the robust behaviors (exhaustion, IFNg secretion, positive cytotoxicity, efficacy ordering at scale) to hard gates?
